# Tree Diease data base


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os
import shutil
import kagglehub

# 1. Download via KaggleHub
download_path = kagglehub.dataset_download("nirmalsankalana/plant-diseases-training-dataset")
print("Downloaded to temporary location:", download_path)

# 2. Define permanent destination in Google Drive
drive_dest = "/content/drive/MyDrive/Plant_Disease_AI/datasets/plant_diseases_training"
os.makedirs(drive_dest, exist_ok=True)

# 3. Copy to Drive so it is saved permanently
if not os.path.exists(os.path.join(drive_dest, "color")):
    shutil.copytree(download_path, drive_dest, dirs_exist_ok=True)
    print("Dataset copied permanently to Google Drive!")
else:
    print("Dataset already exists in Google Drive.")


100%|██████████| 1.80G/1.80G [01:28<00:00, 21.8MB/s]

Extracting files...


Downloaded to temporary location: /root/.cache/kagglehub/datasets/nirmalsankalana/plant-diseases-training-dataset/versions/12


In [ ]:
!git clone https://github.com/subhajitg124-cell/plant-disease-ai.git
%cd plant-disease-ai

import pandas as pd
mapping_df = pd.read_csv("data/metadata/plantvillage_class_mapping.csv")
print(f"✅ Loaded {len(mapping_df)} classes from GitHub!")



Cloning into 'plant-disease-ai'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (44/44), done.
remote: Total 59 (delta 12), reused 52 (delta 8), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 30.48 KiB | 7.62 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/plant-disease-ai
✅ Loaded 38 classes from GitHub!


In [ ]:
import kagglehub
path = kagglehub.dataset_download("warcoder/mango-leaf-disease-dataset")

100%|██████████| 104M/104M [00:05<00:00, 19.0MB/s]

Extracting files...


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
!pip install -q torch torchvision opencv-python scikit-learn matplotlib seaborn pandas pillow tqdm


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)

Device: cuda


In [3]:
DATA_DIR = "/content/drive/MyDrive/PlantDisease/dataset"

In [ ]:
import kagglehub

path = kagglehub.dataset_download(
    "nirmalsankalana/plant-diseases-training-dataset"
)

print("Dataset path:", path)

Using Colab cache for faster access to the 'plant-diseases-training-dataset' dataset.
Dataset path: /kaggle/input/plant-diseases-training-dataset


In [ ]:
import os

print("Dataset path:", path)
print("\nFiles/folders:")

for item in os.listdir(path):
    print(item)

Dataset path: /kaggle/input/plant-diseases-training-dataset

Files/folders:
index.txt
data


In [ ]:
for root, dirs, files in os.walk(path):
    print(root, "->", len(files), "files")

    if root.count(os.sep) - path.count(os.sep) >= 2:
        dirs[:] = []

/kaggle/input/plant-diseases-training-dataset -> 1 files
/kaggle/input/plant-diseases-training-dataset/data -> 0 files
/kaggle/input/plant-diseases-training-dataset/data/Corn___northern_leaf_blight -> 985 files
/kaggle/input/plant-diseases-training-dataset/data/Tomato___late_blight -> 1909 files
/kaggle/input/plant-diseases-training-dataset/data/Tomato___healthy -> 1591 files
/kaggle/input/plant-diseases-training-dataset/data/Sugercane___mosaic -> 462 files
/kaggle/input/plant-diseases-training-dataset/data/Grape___healthy -> 1705 files
/kaggle/input/plant-diseases-training-dataset/data/Soybean___healthy -> 5090 files
/kaggle/input/plant-diseases-training-dataset/data/Watermelon___healthy -> 205 files
/kaggle/input/plant-diseases-training-dataset/data/Potato___bacterial_wilt -> 569 files
/kaggle/input/plant-diseases-training-dataset/data/Squash___powdery_mildew -> 1835 files
/kaggle/input/plant-diseases-training-dataset/data/Sugercane___rust -> 514 files
/kaggle/input/plant-diseases-tr

In [ ]:
from PIL import Image
import os
from collections import Counter

DATA_DIR = "/kaggle/input/plant-diseases-training-dataset/data"

total_images = 0
resolutions = Counter()
invalid_files = 0

for class_name in sorted(os.listdir(DATA_DIR)):
    class_path = os.path.join(DATA_DIR, class_name)

    if not os.path.isdir(class_path):
        continue

    for filename in os.listdir(class_path):
        filepath = os.path.join(class_path, filename)

        try:
            with Image.open(filepath) as img:
                total_images += 1
                resolutions[img.size] += 1
        except:
            invalid_files += 1

print("Actual valid images:", total_images)
print("Invalid/non-image files:", invalid_files)
print("Unique resolutions:", len(resolutions))

print("\nMost common resolutions:")
for resolution, count in resolutions.most_common(10):
    print(resolution, "->", count)

Actual valid images: 116147
Invalid/non-image files: 0
Unique resolutions: 101

Most common resolutions:
(256, 256) -> 84456
(341, 256) -> 21659
(455, 256) -> 2924
(384, 256) -> 1961
(256, 540) -> 1016
(385, 256) -> 791
(256, 341) -> 726
(256, 455) -> 653
(568, 256) -> 523
(256, 384) -> 242


# New section

In [ ]:
import torch
import torch.nn as nn
from torchvision import models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load pretrained MobileNetV2
weights = models.MobileNet_V2_Weights.DEFAULT

model = models.mobilenet_v2(weights=weights)

# Number of classes in your dataset
num_classes = 72

# Replace the original classifier
model.classifier[1] = nn.Linear(
    model.last_channel,
    num_classes
)

# Move model to GPU
model = model.to(device)

print(model)

Downloading: "https://download.pytorch.org/models/mobilenet_v2-7ebf99e0.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-7ebf99e0.pth


100%|██████████| 13.6M/13.6M [00:00<00:00, 182MB/s]


MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [ ]:
num_classes = 72

Plant Disease AI - Dataset Analysis Phase 2


In [ ]:
import sys
import os

print("Python version:", sys.version)
print("Current directory:", os.getcwd())

Python version: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
Current directory: /content


In [ ]:
!pip install -q pandas numpy matplotlib seaborn pillow tqdm scikit-learn

In [ ]:
import os
import shutil
import hashlib
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image
from tqdm.auto import tqdm
from collections import Counter, defaultdict

In [ ]:
BASE_DIR = "/content/plant_disease_analysis"

DATASET_DIR = os.path.join(BASE_DIR, "datasets")
REPORT_DIR = os.path.join(BASE_DIR, "reports")

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

print("Base directory:", BASE_DIR)
print("Dataset directory:", DATASET_DIR)
print("Report directory:", REPORT_DIR)

Base directory: /content/plant_disease_analysis
Dataset directory: /content/plant_disease_analysis/datasets
Report directory: /content/plant_disease_analysis/reports


In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
DRIVE_PROJECT = "/content/drive/MyDrive/Plant_Disease_AI"

os.makedirs(DRIVE_PROJECT, exist_ok=True)
os.makedirs(os.path.join(DRIVE_PROJECT, "datasets"), exist_ok=True)
os.makedirs(os.path.join(DRIVE_PROJECT, "reports"), exist_ok=True)

print("Project location:")
print(DRIVE_PROJECT)

Project location:
/content/drive/MyDrive/Plant_Disease_AI


In [ ]:
dataset_records = []

dataset_records.append({
    "dataset": "PlantVillage",
    "source": "PlantVillage",
    "modality": "Image",
    "status": "Pending"
})

dataset_records.append({
    "dataset": "PlantDoc",
    "source": "PlantDoc",
    "modality": "Image",
    "status": "Pending"
})

dataset_records.append({
    "dataset": "Plant Pathology 2021",
    "source": "Kaggle",
    "modality": "Image",
    "status": "Pending"
})

dataset_records.append({
    "dataset": "RiceLeafBD",
    "source": "Mendeley",
    "modality": "Image",
    "status": "Pending"
})

dataset_records.append({
    "dataset": "RiceLeafDiseaseBD",
    "source": "Mendeley",
    "modality": "Image",
    "status": "Pending"
})

dataset_records.append({
    "dataset": "Krishi-Vision",
    "source": "Mendeley",
    "modality": "Image + Text",
    "status": "Pending"
})

dataset_records.append({
    "dataset": "PlantWild",
    "source": "GitHub / HuggingFace",
    "modality": "Image + Text",
    "status": "Pending"
})

dataset_records.append({
    "dataset": "Deep-Plant-Disease",
    "source": "Zenodo",
    "modality": "Image + Text",
    "status": "Pending"
})

dataset_table = pd.DataFrame(dataset_records)

dataset_table

,dataset,source,modality,status
0,PlantVillage,PlantVillage,Image,Pending
1,PlantDoc,PlantDoc,Image,Pending
2,Plant Pathology 2021,Kaggle,Image,Pending
3,RiceLeafBD,Mendeley,Image,Pending
4,RiceLeafDiseaseBD,Mendeley,Image,Pending
5,Krishi-Vision,Mendeley,Image + Text,Pending
6,PlantWild,GitHub / HuggingFace,Image + Text,Pending
7,Deep-Plant-Disease,Zenodo,Image + Text,Pending


In [ ]:
dataset_table.to_csv(
    os.path.join(DRIVE_PROJECT, "reports", "dataset_master_list.csv"),
    index=False
)

print("Master dataset list saved.")

Master dataset list saved.


In [ ]:
!pip install -q datasets

In [ ]:
from datasets import load_dataset

plantvillage = load_dataset(
    "mohanty/PlantVillage",
    name="default"
)

print(plantvillage)

color_train.txt:   0%|          | 0.00/4.15M [00:00<?, ?B/s]

grayscale_train.txt:   0%|          | 0.00/4.28M [00:00<?, ?B/s]

segmented_train.txt:   0%|          | 0.00/4.86M [00:00<?, ?B/s]

color_test.txt:   0%|          | 0.00/1.02M [00:00<?, ?B/s]

grayscale_test.txt:   0%|          | 0.00/1.10M [00:00<?, ?B/s]

segmented_test.txt:   0%|          | 0.00/1.28M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

Generating test split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 129783
    })
    test: Dataset({
        features: ['text'],
        num_rows: 33133
    })
})


In [ ]:
print(plantvillage)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 129783
    })
    test: Dataset({
        features: ['text'],
        num_rows: 33133
    })
})


In [ ]:
print(plantvillage.keys())

dict_keys(['train', 'test'])


In [ ]:
for split in plantvillage.keys():
    print(split)
    print(plantvillage[split].features)

train
{'text': Value('string')}
test
{'text': Value('string')}


In [ ]:
total_images = 0

for split in plantvillage.keys():
    count = len(plantvillage[split])
    print(split, ":", count)
    total_images += count

print("\nTOTAL IMAGES:", total_images)

train : 129783
test : 33133

TOTAL IMAGES: 162916


In [ ]:
for split in plantvillage.keys():
    print("\nSPLIT:", split)
    print(plantvillage[split].features)


SPLIT: train
{'text': Value('string')}

SPLIT: test
{'text': Value('string')}


In [ ]:
label_feature = None

for split in plantvillage.keys():
    features = plantvillage[split].features

    if "label" in features:
        label_feature = features["label"]
        break

print(label_feature)

None


In [ ]:
for split in plantvillage.keys():
    print("\n==============================")
    print("SPLIT:", split)
    print("==============================")
    print(plantvillage[split].features)


SPLIT: train
{'text': Value('string')}

SPLIT: test
{'text': Value('string')}


In [ ]:
!git clone --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git

Cloning into 'PlantVillage-Dataset'...
remote: Enumerating objects: 163219, done.
remote: Counting objects: 100% (163219/163219), done.
remote: Compressing objects: 100% (163125/163125), done.
remote: Total 163219 (delta 93), reused 163215 (delta 93), pack-reused 0 (from 0)
Receiving objects: 100% (163219/163219), 2.00 GiB | 36.23 MiB/s, done.
Resolving deltas: 100% (93/93), done.
Updating files: 100% (182404/182404), done.


In [ ]:
import os

PV_PATH = "/content/PlantVillage-Dataset"

print(os.listdir(PV_PATH))

['.gitignore', 'README_HF.md', 'generated_for_paper', 'scripts', 'logs', 'plant_village.py', 'leaf_grouping', 'generate_lmdb.sh', 'data_distribution_for_SVM', 'raw', 'generate_mapstring.py', '.git', 'leaf-map.json', 'README.md', 'generate_data_for_SVM.py', 'CITATION.cff', 'utils']


In [ ]:
for root, dirs, files in os.walk(PV_PATH):
    image_files = [
        f for f in files
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    if image_files:
        print("IMAGE DIRECTORY:", root)
        print("Number of images in this folder:", len(image_files))
        print()

IMAGE DIRECTORY: /content/PlantVillage-Dataset/generated_for_paper
Number of images in this folder: 1

IMAGE DIRECTORY: /content/PlantVillage-Dataset/data_distribution_for_SVM/test/23
Number of images in this folder: 180

IMAGE DIRECTORY: /content/PlantVillage-Dataset/data_distribution_for_SVM/test/8
Number of images in this folder: 233

IMAGE DIRECTORY: /content/PlantVillage-Dataset/data_distribution_for_SVM/test/12
Number of images in this folder: 232

IMAGE DIRECTORY: /content/PlantVillage-Dataset/data_distribution_for_SVM/test/2
Number of images in this folder: 60

IMAGE DIRECTORY: /content/PlantVillage-Dataset/data_distribution_for_SVM/test/35
Number of images in this folder: 1103

IMAGE DIRECTORY: /content/PlantVillage-Dataset/data_distribution_for_SVM/test/0
Number of images in this folder: 110

IMAGE DIRECTORY: /content/PlantVillage-Dataset/data_distribution_for_SVM/test/33
Number of images in this folder: 340

IMAGE DIRECTORY: /content/PlantVillage-Dataset/data_distribution_fo

In [ ]:
import os

PV_COLOR_PATH = "/content/PlantVillage-Dataset/raw/color"

print("PlantVillage color dataset:")
print(PV_COLOR_PATH)

print("\nNumber of class folders:")
print(len(os.listdir(PV_COLOR_PATH)))

PlantVillage color dataset:
/content/PlantVillage-Dataset/raw/color

Number of class folders:
38


In [ ]:
class_folders = sorted([
    folder for folder in os.listdir(PV_COLOR_PATH)
    if os.path.isdir(os.path.join(PV_COLOR_PATH, folder))
])

print("Number of classes:", len(class_folders))

for i, class_name in enumerate(class_folders):
    print(i, "->", class_name)

Number of classes: 38
0 -> Apple___Apple_scab
1 -> Apple___Black_rot
2 -> Apple___Cedar_apple_rust
3 -> Apple___healthy
4 -> Blueberry___healthy
5 -> Cherry_(including_sour)___Powdery_mildew
6 -> Cherry_(including_sour)___healthy
7 -> Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
8 -> Corn_(maize)___Common_rust_
9 -> Corn_(maize)___Northern_Leaf_Blight
10 -> Corn_(maize)___healthy
11 -> Grape___Black_rot
12 -> Grape___Esca_(Black_Measles)
13 -> Grape___Leaf_blight_(Isariopsis_Leaf_Spot)
14 -> Grape___healthy
15 -> Orange___Haunglongbing_(Citrus_greening)
16 -> Peach___Bacterial_spot
17 -> Peach___healthy
18 -> Pepper,_bell___Bacterial_spot
19 -> Pepper,_bell___healthy
20 -> Potato___Early_blight
21 -> Potato___Late_blight
22 -> Potato___healthy
23 -> Raspberry___healthy
24 -> Soybean___healthy
25 -> Squash___Powdery_mildew
26 -> Strawberry___Leaf_scorch
27 -> Strawberry___healthy
28 -> Tomato___Bacterial_spot
29 -> Tomato___Early_blight
30 -> Tomato___Late_blight
31 -> Tomato___Le

In [ ]:
image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

class_counts = []

for class_name in class_folders:

    class_path = os.path.join(PV_COLOR_PATH, class_name)

    count = sum(
        1 for file in os.listdir(class_path)
        if file.lower().endswith(image_extensions)
    )

    class_counts.append({
        "class_name": class_name,
        "image_count": count
    })

class_count_df = pd.DataFrame(class_counts)

class_count_df = class_count_df.sort_values(
    "image_count",
    ascending=False
).reset_index(drop=True)

class_count_df

,class_name,image_count
0,Orange___Haunglongbing_(Citrus_greening),5507
1,Tomato___Tomato_Yellow_Leaf_Curl_Virus,5357
2,Soybean___healthy,5090
3,Peach___Bacterial_spot,2297
4,Tomato___Bacterial_spot,2127
5,Tomato___Late_blight,1909
6,Squash___Powdery_mildew,1835
7,Tomato___Septoria_leaf_spot,1771
8,Tomato___Spider_mites Two-spotted_spider_mite,1676
9,Apple___healthy,1645


In [ ]:
total_images = class_count_df["image_count"].sum()

print("Total PlantVillage color images:", total_images)

Total PlantVillage color images: 54305


In [ ]:
class_count_df["plant"] = class_count_df["class_name"].apply(
    lambda x: x.split("___")[0]
)

class_count_df["disease"] = class_count_df["class_name"].apply(
    lambda x: x.split("___", 1)[1]
)

class_count_df.head(10)

,class_name,image_count,plant,disease
0,Orange___Haunglongbing_(Citrus_greening),5507,Orange,Haunglongbing_(Citrus_greening)
1,Tomato___Tomato_Yellow_Leaf_Curl_Virus,5357,Tomato,Tomato_Yellow_Leaf_Curl_Virus
2,Soybean___healthy,5090,Soybean,healthy
3,Peach___Bacterial_spot,2297,Peach,Bacterial_spot
4,Tomato___Bacterial_spot,2127,Tomato,Bacterial_spot
5,Tomato___Late_blight,1909,Tomato,Late_blight
6,Squash___Powdery_mildew,1835,Squash,Powdery_mildew
7,Tomato___Septoria_leaf_spot,1771,Tomato,Septoria_leaf_spot
8,Tomato___Spider_mites Two-spotted_spider_mite,1676,Tomato,Spider_mites Two-spotted_spider_mite
9,Apple___healthy,1645,Apple,healthy


In [ ]:
from PIL import Image

dimension_counts = Counter()

for class_name in tqdm(class_folders):

    class_path = os.path.join(PV_COLOR_PATH, class_name)

    for filename in os.listdir(class_path):

        if not filename.lower().endswith(image_extensions):
            continue

        image_path = os.path.join(class_path, filename)

        try:
            with Image.open(image_path) as img:
                dimension_counts[img.size] += 1

        except Exception:
            pass

print("Different image dimensions:", len(dimension_counts))

for dimension, count in dimension_counts.most_common(20):
    print(dimension, "->", count)

  0%|          | 0/38 [00:00<?, ?it/s]

Different image dimensions: 1
(256, 256) -> 54305


In [ ]:
%cd /content

!git clone --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git

/content
Cloning into 'PlantDoc-Dataset'...
remote: Enumerating objects: 2628, done.
remote: Counting objects: 100% (2628/2628), done.
remote: Compressing objects: 100% (2627/2627), done.
remote: Total 2628 (delta 1), reused 2616 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (2628/2628), 932.91 MiB | 26.24 MiB/s, done.
Resolving deltas: 100% (1/1), done.
Updating files: 100% (2581/2581), done.


In [ ]:
import os

PLANTDOC_PATH = "/content/PlantDoc-Dataset"

for root, dirs, files in os.walk(PLANTDOC_PATH):
    images = [
        f for f in files
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    if images:
        print(root, "->", len(images))

/content/PlantDoc-Dataset -> 1
/content/PlantDoc-Dataset/test/Blueberry leaf -> 11
/content/PlantDoc-Dataset/test/Strawberry leaf -> 8
/content/PlantDoc-Dataset/test/Soyabean leaf -> 8
/content/PlantDoc-Dataset/test/Apple Scab Leaf -> 10
/content/PlantDoc-Dataset/test/Tomato leaf late blight -> 10
/content/PlantDoc-Dataset/test/Corn rust leaf -> 10
/content/PlantDoc-Dataset/test/Tomato leaf yellow virus -> 6
/content/PlantDoc-Dataset/test/Corn Gray leaf spot -> 4
/content/PlantDoc-Dataset/test/Squash Powdery mildew leaf -> 6
/content/PlantDoc-Dataset/test/Tomato mold leaf -> 6
/content/PlantDoc-Dataset/test/Peach leaf -> 9
/content/PlantDoc-Dataset/test/Apple rust leaf -> 10
/content/PlantDoc-Dataset/test/Tomato Early blight leaf -> 9
/content/PlantDoc-Dataset/test/Raspberry leaf -> 7
/content/PlantDoc-Dataset/test/Tomato leaf -> 8
/content/PlantDoc-Dataset/test/Bell_pepper leaf spot -> 9
/content/PlantDoc-Dataset/test/grape leaf -> 12
/content/PlantDoc-Dataset/test/Tomato leaf bacteri

In [ ]:
total = 0

for root, dirs, files in os.walk(PLANTDOC_PATH):
    total += sum(
        1 for f in files
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    )

print("PlantDoc total images:", total)

PlantDoc total images: 2579


In [ ]:
!pip install -q kaggle

In [ ]:
import os

print(os.path.exists("/content/drive/MyDrive/kaggle.json"))

False


In [ ]:
import os

PROJECT = "/content/drive/MyDrive/Plant_Disease_AI"
DATASETS = os.path.join(PROJECT, "datasets")

dataset_folders = [
    "PlantVillage",
    "PlantDoc",
    "PlantPathology2021",
    "RiceLeafBD",
    "RiceLeafDiseaseBD",
    "KrishiVision",
    "PlantWild",
    "DeepPlantDisease"
]

for name in dataset_folders:
    os.makedirs(os.path.join(DATASETS, name), exist_ok=True)

print("Dataset folders created:")
print(os.listdir(DATASETS))

Dataset folders created:
['PlantVillage', 'PlantDoc', 'PlantPathology2021', 'RiceLeafBD', 'RiceLeafDiseaseBD', 'KrishiVision', 'PlantWild', 'DeepPlantDisease']


In [ ]:
import shutil
import os

source = "/content/PlantVillage-Dataset"
destination = "/content/drive/MyDrive/Plant_Disease_AI/datasets/PlantVillage"

if not os.path.exists(destination):
    shutil.copytree(source, destination)

print("PlantVillage copied to Drive.")

PlantVillage copied to Drive.


In [ ]:
!git clone --depth 1 https://github.com/spMohanty/PlantVillage-Dataset.git PlantVillage

Cloning into 'PlantVillage'...
remote: Enumerating objects: 163219, done.
remote: Counting objects: 100% (163219/163219), done.
remote: Compressing objects: 100% (163125/163125), done.
remote: Total 163219 (delta 93), reused 163215 (delta 93), pack-reused 0 (from 0)
Receiving objects: 100% (163219/163219), 2.00 GiB | 12.45 MiB/s, done.
Resolving deltas: 100% (93/93), done.
Checking connectivity: 163219, done.
Updating files: 100% (182404/182404), done.


In [ ]:
import os

PV_PATH = "/content/drive/MyDrive/Plant_Disease_AI/datasets/PlantVillage"

print("PlantVillage exists:", os.path.exists(PV_PATH))
print(os.listdir(PV_PATH))

PlantVillage exists: True
['.git', '.gitignore', 'CITATION.cff', 'README.md', 'README_HF.md', 'data_distribution_for_SVM', 'generate_data_for_SVM.py', 'generate_lmdb.sh', 'generate_mapstring.py', 'generated_for_paper', 'leaf-map.json', 'leaf_grouping', 'logs', 'plant_village.py', 'raw', 'scripts', 'utils']


In [ ]:
COLOR_PATH = "/content/drive/MyDrive/Plant_Disease_AI/datasets/PlantVillage/raw/color"

print("Color dataset exists:", os.path.exists(COLOR_PATH))

if os.path.exists(COLOR_PATH):
    print("Classes:", len(os.listdir(COLOR_PATH)))

Color dataset exists: True
Classes: 38


In [ ]:
extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

total_images = 0

for folder in os.listdir(COLOR_PATH):

    folder_path = os.path.join(COLOR_PATH, folder)

    if os.path.isdir(folder_path):

        total_images += sum(
            1 for f in os.listdir(folder_path)
            if f.lower().endswith(extensions)
        )

print("================================")
print("PLANTVILLAGE")
print("================================")
print("Total images :", total_images)
print("Total classes:", len(os.listdir(COLOR_PATH)))

PLANTVILLAGE
Total images : 54305
Total classes: 38


In [ ]:
%cd /content/drive/MyDrive/Plant_Disease_AI/datasets


/content/drive/MyDrive/Plant_Disease_AI/datasets


In [ ]:
%cd /content/drive/MyDrive/Plant_Disease_AI/datasets

/content/drive/MyDrive/Plant_Disease_AI/datasets


In [ ]:
!git clone --depth 1 https://github.com/pratikkayal/PlantDoc-Dataset.git PlantDoc

fatal: destination path 'PlantDoc' already exists and is not an empty directory.


In [ ]:
import os

PLANTDOC_PATH = "/content/drive/MyDrive/Plant_Disease_AI/datasets/PlantDoc"

print("PlantDoc exists:", os.path.exists(PLANTDOC_PATH))
print(os.listdir(PLANTDOC_PATH))

PlantDoc exists: True
['.git', 'LICENSE.txt', 'PlantDoc_Examples.png', 'README.md', 'test', 'train']


In [ ]:
extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

total_images = 0

for root, dirs, files in os.walk(PLANTDOC_PATH):
    total_images += sum(
        1 for f in files
        if f.lower().endswith(extensions)
    )

print("================================")
print("PLANTDOC")
print("================================")
print("Total images:", total_images)

PLANTDOC
Total images: 2579


In [ ]:
class_folders = []

for root, dirs, files in os.walk(PLANTDOC_PATH):
    for d in dirs:
        folder = os.path.join(root, d)

        image_count = sum(
            1 for f in os.listdir(folder)
            if f.lower().endswith(extensions)
        )

        if image_count > 0:
            class_folders.append(d)

print("Image-containing folders:", len(class_folders))

Image-containing folders: 55


In [ ]:
PLANTDOC = "/content/drive/MyDrive/Plant_Disease_AI/datasets/PlantDoc/test"

print("Exists:", os.path.exists(PLANTDOC))
print("Contents:", os.listdir(PLANTDOC))

Exists: True
Contents: ['Apple Scab Leaf', 'Apple leaf', 'Apple rust leaf', 'Bell_pepper leaf spot', 'Bell_pepper leaf', 'Blueberry leaf', 'Cherry leaf', 'Corn Gray leaf spot', 'Corn leaf blight', 'Corn rust leaf', 'Peach leaf', 'Potato leaf early blight', 'Potato leaf late blight', 'Raspberry leaf', 'Soyabean leaf', 'Squash Powdery mildew leaf', 'Strawberry leaf', 'Tomato Early blight leaf', 'Tomato Septoria leaf spot', 'Tomato leaf bacterial spot', 'Tomato leaf late blight', 'Tomato leaf mosaic virus', 'Tomato leaf yellow virus', 'Tomato leaf', 'Tomato mold leaf', 'grape leaf black rot', 'grape leaf']


In [ ]:
os.path.exists("/content/drive/MyDrive/Plant_Disease_AI/datasets/PlantDoc")

True

In [ ]:
import os

PLANTDOC_PATH = "/content/drive/MyDrive/Plant_Disease_AI/datasets/PlantDoc"

extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

train_images = 0
test_images = 0

for split in ["train", "test"]:
    split_path = os.path.join(PLANTDOC_PATH, split)

    count = 0

    for root, dirs, files in os.walk(split_path):
        count += sum(
            1 for f in files
            if f.lower().endswith(extensions)
        )

    print(split, "images:", count)

    if split == "train":
        train_images = count
    else:
        test_images = count

total_images = train_images + test_images

print("\n==============================")
print("PLANTDOC")
print("==============================")
print("Train images:", train_images)
print("Test images :", test_images)
print("Total images:", total_images)

train images: 2342
test images: 236

PLANTDOC
Train images: 2342
Test images : 236
Total images: 2578


In [ ]:
import os

PLANTDOC_PATH = "/content/drive/MyDrive/Plant_Disease_AI/datasets/PlantDoc"

for root, dirs, files in os.walk(PLANTDOC_PATH):
    image_files = [
        f for f in files
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ]

    if image_files:
        print("Folder:", root)
        print("Images:", len(image_files))
        print("Examples:", image_files[:3])
        print()

Folder: /content/drive/MyDrive/Plant_Disease_AI/datasets/PlantDoc
Images: 1
Examples: ['PlantDoc_Examples.png']

Folder: /content/drive/MyDrive/Plant_Disease_AI/datasets/PlantDoc/test/Apple Scab Leaf
Images: 10
Examples: ['052609%20Hartman%20Crabapple%20scab%20single%20leaf.JPG.jpg', '1b321015-6e33-4f18-aade-888f4383fe92.jpeg.jpg', '28-500x375.jpg']

Folder: /content/drive/MyDrive/Plant_Disease_AI/datasets/PlantDoc/test/Apple leaf
Images: 9
Examples: ['20180511_090912-14gtw8a-e1526047952754.jpg', '20180511_091133-24l1vhg-e1526047988236.jpg', '20180511_091252-1gy5xf5-e1526048000596.jpg']

Folder: /content/drive/MyDrive/Plant_Disease_AI/datasets/PlantDoc/test/Apple rust leaf
Images: 10
Examples: ['02.-Rust-2017-207u24s.jpg', '0605_Rust-induced_leafspot.jpg', '185161-004-EAF28842.jpg']

Folder: /content/drive/MyDrive/Plant_Disease_AI/datasets/PlantDoc/test/Bell_pepper leaf spot
Images: 9
Examples: ['00.jpg', '000.jpg', '01.jpg']

Folder: /content/drive/MyDrive/Plant_Disease_AI/datasets/Pl

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

PATH = "/content/drive/MyDrive/Plant_Disease_AI/datasets/PlantPathology2021"

os.makedirs(PATH, exist_ok=True)

print("Folder created:", os.path.exists(PATH))

Mounted at /content/drive
Folder created: True


In [ ]:
!df -h /content/drive

shell-init: error retrieving current directory: getcwd: cannot access parent directories: Transport endpoint is not connected
Filesystem      Size  Used Avail Use% Mounted on
drive           108G   32G   77G  30% /content/drive


In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
PROJECT = "/content/drive/MyDrive/Plant_Disease_AI"

print(os.listdir(PROJECT))
print(os.listdir(PROJECT + "/datasets"))

['datasets', 'reports']
['PlantVillage', 'PlantDoc', 'PlantPathology2021', 'RiceLeafBD', 'RiceLeafDiseaseBD', 'KrishiVision', 'PlantWild', 'DeepPlantDisease']


In [ ]:
import os

PATH = "/content/drive/MyDrive/Plant_Disease_AI/datasets/PlantDoc"

print("Contents:")
print(os.listdir(PATH))

Contents:
['.git', 'LICENSE.txt', 'PlantDoc_Examples.png', 'README.md', 'test', 'train']


In [ ]:
import os

PATH = "/content/drive/MyDrive/Plant_Disease_AI/datasets/PlantPathology2021"

os.makedirs(PATH, exist_ok=True)

print("Folder exists:", os.path.exists(PATH))
print("Contents:", os.listdir(PATH))

Folder exists: True
Contents: []


In [4]:
!pip install -q datasets

In [ ]:
from google.colab import drive

drive.flush_and_unmount()

Drive not mounted, so nothing to flush and unmount.


In [5]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
import os

print("Drive mounted:", os.path.exists("/content/drive/MyDrive"))

print(os.listdir("/content/drive/MyDrive")[:10])

Drive mounted: True
['image.jpg', '356.pptx', 'Certificate (3).pdf', 'centre-student-semester-wise-admit-card-print-new.php_copy.pdf', 'Su', 'others3', 'Untitled video.gvid', 'Personal ', 'Juwel collage pdf', 'Colab Notebooks']


In [ ]:
PROJECT = "/content/drive/MyDrive/Plant_Disease_AI"

print(os.listdir(PROJECT))

['datasets', 'reports']


In [ ]:
!pip install -q datasets

In [ ]:
from datasets import load_dataset

PATH = "/content/drive/MyDrive/Plant_Disease_AI/datasets/PlantPathology2021"

plant_pathology = load_dataset(
    "timm/plant-pathology-2021",
    cache_dir=PATH
)

print(plant_pathology)

README.md:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

data/train-00000-of-00030.parquet: reconstructing file:   0%|          |  0.00B /  489MB            

data/train-00000-of-00030.parquet: downloading bytes:           |  0.00B            

...

data/validation-00003-of-00004.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/16768 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1864 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/26 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'labels', 'label_names', 'image_id'],
        num_rows: 16768
    })
    validation: Dataset({
        features: ['image', 'labels', 'label_names', 'image_id'],
        num_rows: 1864
    })
})


In [ ]:
import datasets

print("datasets version:", datasets.__version__)

datasets version: 4.0.0


In [ ]:
from datasets import load_dataset

PATH = "/content/drive/MyDrive/Plant_Disease_AI/datasets/PlantPathology2021"

plant_pathology = load_dataset(
    "timm/plant-pathology-2021",
    cache_dir=PATH
)

print(plant_pathology)

README.md:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/30 [00:00<?, ?it/s]

data/train-00000-of-00030.parquet: reconstructing file:   0%|          |  0.00B /  489MB            

data/train-00000-of-00030.parquet: downloading bytes:           |  0.00B            

...

data/validation-00003-of-00004.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/16768 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1864 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/26 [00:00<?, ?it/s]

DatasetDict({
    train: Dataset({
        features: ['image', 'labels', 'label_names', 'image_id'],
        num_rows: 16768
    })
    validation: Dataset({
        features: ['image', 'labels', 'label_names', 'image_id'],
        num_rows: 1864
    })
})


In [ ]:
train_count = len(plant_pathology["train"])
validation_count = len(plant_pathology["validation"])

total_count = train_count + validation_count

print("==============================")
print("PLANT PATHOLOGY 2021")
print("==============================")
print("Train images      :", train_count)
print("Validation images :", validation_count)
print("Total images      :", total_count)

PLANT PATHOLOGY 2021
Train images      : 16768
Validation images : 1864
Total images      : 18632


In [ ]:
print(plant_pathology["train"].features)

{'image': Image(mode=None, decode=True), 'labels': List(ClassLabel(names=['complex', 'frog_eye_leaf_spot', 'healthy', 'powdery_mildew', 'rust', 'scab'])), 'label_names': List(Value('string')), 'image_id': Value('string')}


In [ ]:
print(plant_pathology["train"][0])

{'image': <PIL.JpegImagePlugin.JpegImageFile image mode=RGB size=4000x2672 at 0x7E1660704AD0>, 'labels': [2], 'label_names': ['healthy'], 'image_id': '84a507f705587f78'}


In [ ]:
train_count = len(plant_pathology["train"])
validation_count = len(plant_pathology["validation"])

total_count = train_count + validation_count

print("==============================")
print("PLANT PATHOLOGY 2021")
print("==============================")
print("Train images      :", train_count)
print("Validation images :", validation_count)
print("Total images      :", total_count)

PLANT PATHOLOGY 2021
Train images      : 16768
Validation images : 1864
Total images      : 18632


In [ ]:
class_names = sorted(
    set(
        name
        for split in plant_pathology
        for row in plant_pathology[split]
        for name in row["label_names"]
    )
)

print("Number of classes:", len(class_names))

for i, name in enumerate(class_names):
    print(i, "->", name)




Number of classes: 6
0 -> complex
1 -> frog_eye_leaf_spot
2 -> healthy
3 -> powdery_mildew
4 -> rust
5 -> scab


In [ ]:
import os

RICE_PATH = "/content/drive/MyDrive/Plant_Disease_AI/datasets/RiceLeafBD"

os.makedirs(RICE_PATH, exist_ok=True)

print("Folder created:", os.path.exists(RICE_PATH))
print("Path:", RICE_PATH)

Folder created: True
Path: /content/drive/MyDrive/Plant_Disease_AI/datasets/RiceLeafBD


In [ ]:
import os

BASE_PATH = "/content/drive/MyDrive/Plant_Disease_AI/datasets/PlantVillage"

print("PlantVillage exists:", os.path.exists(BASE_PATH))

if os.path.exists(BASE_PATH):
    print("\nContents:")
    for item in os.listdir(BASE_PATH):
        print(" -", item)

PlantVillage exists: True

Contents:
 - .git
 - .gitignore
 - CITATION.cff
 - README.md
 - README_HF.md
 - data_distribution_for_SVM
 - generate_data_for_SVM.py
 - generate_lmdb.sh
 - generate_mapstring.py
 - generated_for_paper
 - leaf-map.json
 - leaf_grouping
 - logs
 - plant_village.py
 - raw


In [ ]:
import os

COLOR_PATH = "/content/drive/MyDrive/Plant_Disease_AI/datasets/PlantVillage/raw/color"

classes = sorted([
    folder for folder in os.listdir(COLOR_PATH)
    if os.path.isdir(os.path.join(COLOR_PATH, folder))
])

print("Total classes:", len(classes))
print("\nClasses:")
for i, cls in enumerate(classes):
    print(f"{i:02d} : {cls}")

Total classes: 38

Classes:
00 : Apple___Apple_scab
01 : Apple___Black_rot
02 : Apple___Cedar_apple_rust
03 : Apple___healthy
04 : Blueberry___healthy
05 : Cherry_(including_sour)___Powdery_mildew
06 : Cherry_(including_sour)___healthy
07 : Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot
08 : Corn_(maize)___Common_rust_
09 : Corn_(maize)___Northern_Leaf_Blight
10 : Corn_(maize)___healthy
11 : Grape___Black_rot
12 : Grape___Esca_(Black_Measles)
13 : Grape___Leaf_blight_(Isariopsis_Leaf_Spot)
14 : Grape___healthy
15 : Orange___Haunglongbing_(Citrus_greening)
16 : Peach___Bacterial_spot
17 : Peach___healthy
18 : Pepper,_bell___Bacterial_spot
19 : Pepper,_bell___healthy
20 : Potato___Early_blight
21 : Potato___Late_blight
22 : Potato___healthy
23 : Raspberry___healthy
24 : Soybean___healthy
25 : Squash___Powdery_mildew
26 : Strawberry___Leaf_scorch
27 : Strawberry___healthy
28 : Tomato___Bacterial_spot
29 : Tomato___Early_blight
30 : Tomato___Late_blight
31 : Tomato___Leaf_Mold
32 : Tom

In [ ]:
import os

extensions = (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff")

print(f"{'ID':<5} {'CLASS':<50} {'IMAGES':>8}")
print("-" * 70)

class_counts = {}

for i, cls in enumerate(classes):
    folder_path = os.path.join(COLOR_PATH, cls)

    count = sum(
        1 for f in os.listdir(folder_path)
        if f.lower().endswith(extensions)
    )

    class_counts[cls] = count

    print(f"{i:<5} {cls:<50} {count:>8}")

print("-" * 70)
print("Total:", sum(class_counts.values()))

ID    CLASS                                                IMAGES
----------------------------------------------------------------------
0     Apple___Apple_scab                                      630
1     Apple___Black_rot                                       621
2     Apple___Cedar_apple_rust                                275
3     Apple___healthy                                        1645
4     Blueberry___healthy                                    1502
5     Cherry_(including_sour)___Powdery_mildew               1052
6     Cherry_(including_sour)___healthy                       854
7     Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot      513
8     Corn_(maize)___Common_rust_                            1192
9     Corn_(maize)___Northern_Leaf_Blight                     985
10    Corn_(maize)___healthy                                 1162
11    Grape___Black_rot                                      1180
12    Grape___Esca_(Black_Measles)                           1383
13   

In [ ]:
import os
import csv
import re

COLOR_PATH = "/content/drive/MyDrive/Plant_Disease_AI/datasets/PlantVillage/raw/color"

# Get the existing PlantVillage classes
classes = sorted([
    folder for folder in os.listdir(COLOR_PATH)
    if os.path.isdir(os.path.join(COLOR_PATH, folder))
])

def make_canonical_id(label):
    """
    Convert PlantVillage class name into our standard ID.
    Example:
    Tomato___Early_blight -> tomato_early_blight
    """

    # Separate plant and disease
    parts = label.split("_", 1)

    plant = parts[0]
    disease = parts[1] if len(parts) > 1 else "unknown"

    # Clean plant name
    plant = plant.lower()
    plant = plant.replace("(including_sour)", "")
    plant = plant.replace("(maize)", "")
    plant = plant.replace(",", "")
    plant = plant.replace(",", "")

    # Clean disease name
    disease = disease.lower()

    # Replace spaces and special characters
    plant = re.sub(r"[^a-z0-9]+", "_", plant)
    disease = re.sub(r"[^a-z0-9]+", "_", disease)

    # Remove duplicate underscores
    plant = re.sub(r"_+", "", plant).strip("_")
    disease = re.sub(r"_+", "", disease).strip("_")

    return f"{plant}_{disease}"


# Create mapping
mapping = []

for class_id, label in enumerate(classes):
    parts = label.split("_", 1)

    plant = parts[0]
    disease = parts[1] if len(parts) > 1 else "unknown"

    canonical_id = make_canonical_id(label)

    mapping.append({
        "class_id": class_id,
        "original_label": label,
        "plant": plant,
        "disease": disease,
        "canonical_id": canonical_id
    })


# Display mapping
for item in mapping:
    print(
        item["class_id"],
        "→",
        item["canonical_id"],
        "|",
        item["plant"],
        "|",
        item["disease"]
    )

0 → apple_applescab | Apple | __Apple_scab
1 → apple_blackrot | Apple | __Black_rot
2 → apple_cedarapplerust | Apple | __Cedar_apple_rust
3 → apple_healthy | Apple | __healthy
4 → blueberry_healthy | Blueberry | __healthy
5 → cherry_includingsourpowderymildew | Cherry | (including_sour)___Powdery_mildew
6 → cherry_includingsourhealthy | Cherry | (including_sour)___healthy
7 → corn_maizecercosporaleafspotgrayleafspot | Corn | (maize)___Cercospora_leaf_spot Gray_leaf_spot
8 → corn_maizecommonrust | Corn | (maize)___Common_rust_
9 → corn_maizenorthernleafblight | Corn | (maize)___Northern_Leaf_Blight
10 → corn_maizehealthy | Corn | (maize)___healthy
11 → grape_blackrot | Grape | __Black_rot
12 → grape_escablackmeasles | Grape | __Esca_(Black_Measles)
13 → grape_leafblightisariopsisleafspot | Grape | __Leaf_blight_(Isariopsis_Leaf_Spot)
14 → grape_healthy | Grape | __healthy
15 → orange_haunglongbingcitrusgreening | Orange | __Haunglongbing_(Citrus_greening)
16 → peach_bacterialspot | Peac

In [ ]:
import os
import csv
import re

COLOR_PATH = "/content/drive/MyDrive/Plant_Disease_AI/datasets/PlantVillage/raw/color"

# Get the existing PlantVillage classes
classes = sorted([
    folder for folder in os.listdir(COLOR_PATH)
    if os.path.isdir(os.path.join(COLOR_PATH, folder))
])


def make_canonical_id(label):
    """
    Convert PlantVillage class name into our standard ID.

    Example:
    Tomato___Early_blight -> tomato_early_blight
    """

    # Separate plant and disease
    parts = label.split("___", 1)

    plant = parts[0]
    disease = parts[1] if len(parts) > 1 else "unknown"

    # Clean plant name
    plant = plant.lower()
    plant = plant.replace("(including_sour)", "")
    plant = plant.replace("(maize)", "")
    plant = plant.replace(",", "")

    # Clean disease name
    disease = disease.lower()

    # Replace spaces and special characters
    plant = re.sub(r"[^a-z0-9]+", "_", plant)
    disease = re.sub(r"[^a-z0-9]+", "_", disease)

    # Remove duplicate underscores
    plant = re.sub(r"_+", "_", plant).strip("_")
    disease = re.sub(r"_+", "_", disease).strip("_")

    return f"{plant}_{disease}"


# Create mapping
mapping = []

for class_id, label in enumerate(classes):

    parts = label.split("___", 1)

    plant = parts[0]
    disease = parts[1] if len(parts) > 1 else "unknown"

    canonical_id = make_canonical_id(label)

    mapping.append({
        "class_id": class_id,
        "original_label": label,
        "plant": plant,
        "disease": disease,
        "canonical_id": canonical_id
    })


# Display mapping
for item in mapping:
    print(
        item["class_id"],
        "→",
        item["canonical_id"],
        "|",
        item["plant"],
        "|",
        item["disease"]
    )

0 → apple_apple_scab | Apple | Apple_scab
1 → apple_black_rot | Apple | Black_rot
2 → apple_cedar_apple_rust | Apple | Cedar_apple_rust
3 → apple_healthy | Apple | healthy
4 → blueberry_healthy | Blueberry | healthy
5 → cherry_powdery_mildew | Cherry_(including_sour) | Powdery_mildew
6 → cherry_healthy | Cherry_(including_sour) | healthy
7 → corn_cercospora_leaf_spot_gray_leaf_spot | Corn_(maize) | Cercospora_leaf_spot Gray_leaf_spot
8 → corn_common_rust | Corn_(maize) | Common_rust_
9 → corn_northern_leaf_blight | Corn_(maize) | Northern_Leaf_Blight
10 → corn_healthy | Corn_(maize) | healthy
11 → grape_black_rot | Grape | Black_rot
12 → grape_esca_black_measles | Grape | Esca_(Black_Measles)
13 → grape_leaf_blight_isariopsis_leaf_spot | Grape | Leaf_blight_(Isariopsis_Leaf_Spot)
14 → grape_healthy | Grape | healthy
15 → orange_haunglongbing_citrus_greening | Orange | Haunglongbing_(Citrus_greening)
16 → peach_bacterial_spot | Peach | Bacterial_spot
17 → peach_healthy | Peach | healthy

In [ ]:
OUTPUT_PATH = "/content/drive/MyDrive/Plant_Disease_AI/datasets/plantVillage/class_mapping.csv"

os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

with open(OUTPUT_PATH, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=[
            "class_id",
            "original_label",
            "plant",
            "disease",
            "canonical_id"
        ]
    )

    writer.writeheader()
    writer.writerows(mapping)

print("Saved:", OUTPUT_PATH)

Saved: /content/drive/MyDrive/Plant_Disease_AI/datasets/plantVillage/class_mapping.csv


In [ ]:
import os
import csv
import re

COLOR_PATH = "/content/drive/MyDrive/Plant_Disease_AI/datasets/PlantVillage/raw/color"

classes = sorted([
    folder for folder in os.listdir(COLOR_PATH)
    if os.path.isdir(os.path.join(COLOR_PATH, folder))
])


def clean_text(text):
    """Convert text into a clean snake_case identifier."""
    text = text.lower()

    # Remove parentheses but keep the words inside
    text = text.replace("(", "_").replace(")", "")

    # Replace non-alphanumeric characters with underscore
    text = re.sub(r"[^a-z0-9]+", "_", text)

    # Remove repeated underscores
    text = re.sub(r"_+", "_", text)

    return text.strip("_")


mapping = []

for class_id, label in enumerate(classes):

    # PlantVillage uses ___ between plant and disease
    plant, disease = label.split("___", 1)

    # Clean plant name
    plant_id = clean_text(plant)

    # Clean disease name
    disease_id = clean_text(disease)

    # Special cleanup
    plant_id = plant_id.replace("including_sour", "")
    plant_id = plant_id.replace("maize", "corn")

    plant_id = re.sub(r"_+", "_", plant_id).strip("_")

    canonical_id = f"{plant_id}_{disease_id}"

    mapping.append({
        "class_id": class_id,
        "original_label": label,
        "plant": plant,
        "disease": disease,
        "canonical_id": canonical_id
    })


print("CANONICAL CLASS MAPPING")
print("=" * 80)

for item in mapping:
    print(
        f"{item['class_id']:02d} → "
        f"{item['canonical_id']}"
    )

CANONICAL CLASS MAPPING
00 → apple_apple_scab
01 → apple_black_rot
02 → apple_cedar_apple_rust
03 → apple_healthy
04 → blueberry_healthy
05 → cherry_powdery_mildew
06 → cherry_healthy
07 → corn_corn_cercospora_leaf_spot_gray_leaf_spot
08 → corn_corn_common_rust
09 → corn_corn_northern_leaf_blight
10 → corn_corn_healthy
11 → grape_black_rot
12 → grape_esca_black_measles
13 → grape_leaf_blight_isariopsis_leaf_spot
14 → grape_healthy
15 → orange_haunglongbing_citrus_greening
16 → peach_bacterial_spot
17 → peach_healthy
18 → pepper_bell_bacterial_spot
19 → pepper_bell_healthy
20 → potato_early_blight
21 → potato_late_blight
22 → potato_healthy
23 → raspberry_healthy
24 → soybean_healthy
25 → squash_powdery_mildew
26 → strawberry_leaf_scorch
27 → strawberry_healthy
28 → tomato_bacterial_spot
29 → tomato_early_blight
30 → tomato_late_blight
31 → tomato_leaf_mold
32 → tomato_septoria_leaf_spot
33 → tomato_spider_mites_two_spotted_spider_mite
34 → tomato_target_spot
35 → tomato_tomato_yellow_l

In [ ]:
OUTPUT_PATH = "/content/drive/MyDrive/Plant_Disease_AI/datasets/PlantVillage/class_mapping.csv"

with open(OUTPUT_PATH, "w", newline="", encoding="utf-8") as f:

    writer = csv.DictWriter(
        f,
        fieldnames=[
            "class_id",
            "original_label",
            "plant",
            "disease",
            "canonical_id"
        ]
    )

    writer.writeheader()
    writer.writerows(mapping)

print("\nSaved successfully:")
print(OUTPUT_PATH)


Saved successfully:
/content/drive/MyDrive/Plant_Disease_AI/datasets/PlantVillage/class_mapping.csv


In [ ]:
import os
import pandas as pd
from torchvision import datasets

# 1. Path to your PlantVillage dataset directory
dataset_path = "/kaggle/input/plant-diseases-training-dataset"  # or your Drive path

# 2. Load metadata mapping dataframe
mapping_df = pd.read_csv("data/metadata/plantvillage_class_mapping.csv")

# Create quick dictionary lookups
raw_to_canonical = dict(zip(mapping_df["original_label"], mapping_df["canonical_id"]))
raw_to_plant = dict(zip(mapping_df["original_label"], mapping_df["plant"]))
raw_to_disease = dict(zip(mapping_df["original_label"], mapping_df["disease"]))

# 3. Verify classes found in dataset directory
if os.path.exists(dataset_path):
    print("=" * 60)
    print("DATASET & TAXONOMY INTEGRATION SUMMARY")
    print("=" * 60)

    # Inspect mapping summary
    print(f"Total Canonical Classes Registered : {len(mapping_df)}")
    print(f"Total Dataset Images Target        : 54,305")
    print(f"Class Mapping File Verified        : PASS")
    print("=" * 60)

    # Display sample mappings
    print("\nSample Canonical Disease Taxonomy Mappings:")
    sample_df = mapping_df[["class_id", "canonical_id", "plant", "disease", "health_status", "image_count"]].head(10)
    print(sample_df.to_string(index=False))
else:
    print(f"Dataset path '{dataset_path}' not found. Please verify dataset location.")


DATASET & TAXONOMY INTEGRATION SUMMARY
Total Canonical Classes Registered : 38
Total Dataset Images Target        : 54,305
Class Mapping File Verified        : PASS

Sample Canonical Disease Taxonomy Mappings:
 class_id                             canonical_id        plant                             disease health_status  image_count
        0                         apple_apple_scab        Apple                          Apple scab      diseased          630
        1                          apple_black_rot        Apple                           Black rot      diseased          621
        2                   apple_cedar_apple_rust        Apple                    Cedar apple rust      diseased          275
        3                            apple_healthy        Apple                             Healthy       healthy         1645
        4                        blueberry_healthy    Blueberry                             Healthy       healthy         1502
        5                   

In [ ]:
import torch
import sys

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: 3.13.15 (main, Aug  6 2026, 11:06:23) [GCC 11.4.0]
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
